# 🔍 Notebook 01: Data Profiling & Quality Exploration
**โครงการ**: Used Car Analytics (Data Warehouse & ETL Pipeline)
**กลุ่ม**: GroupXX

---

## 📌 วัตถุประสงค์ของ Notebook นี้:
1. สำรวจและนำเข้าข้อมูลดิบจาก **3 Data Sources หลัก** แยกตามประเภทและแหล่งที่มาอย่างเป็นระบบ
2. สำรวจโครงสร้าง ชนิดข้อมูล (Data Types) และสถิติเบื้องต้นของแต่ละ Data Source
3. ตรวจสอบปัญหาคุณภาพข้อมูล (Data Quality Issues) 5 ประเด็นสำคัญตามเกณฑ์อาจารย์

## 📦 Data Source 1: Kaidee Auto (Nested JSON Data Source)
- **ลักษณะข้อมูล**: ข้อมูลประกาศขายรถมือสองในไทยจากเว็บ Kaidee Auto
- **รูปแบบไฟล์**: `kaidee_cars_detail.json` (Nested JSON Data)
- **ความซับซ้อนตามเกณฑ์อาจารย์**: `JSON / Nested JSON Data`

In [6]:
import pandas as pd
import json
import glob
import os

# นำเข้า Data Source 1: Kaidee Auto JSON
json_path = '../../01_Raw_Data/kaidee/kaidee_cars_detail.json'
with open(json_path, 'r', encoding='utf-8') as f:
    kaidee_raw = json.load(f)

df_kaidee = pd.DataFrame(kaidee_raw)
print(f'✅ [Data Source 1: Kaidee Auto] โหลดสำเร็จ: {len(df_kaidee):,} แถว | {len(df_kaidee.columns)} คอลัมน์')
print('รายชื่อคอลัมน์:', df_kaidee.columns.tolist())
df_kaidee[['id', 'title', 'brand', 'model', 'year', 'price', 'mileage', 'location']].head(5)

✅ [Data Source 1: Kaidee Auto] โหลดสำเร็จ: 1,443 แถว | 24 คอลัมน์
รายชื่อคอลัมน์: ['id', 'title', 'price', 'brand', 'model', 'sub_model', 'year', 'transmission', 'gas_status', 'color', 'fuel_type', 'car_type', 'sub_car_type', 'mileage', 'location', 'seller_name', 'seller_role', 'seller_phone', 'seller_line', 'seller_email', 'description', 'first_approved_time', 'url', 'image_url']


,id,title,brand,model,year,price,mileage,location
0,58e0376d-2372-4740-afed-dc39f84a5ede,F44 220i Gran Coupe Sport CBU ปี 22 2.0 192hp ...,BMW,Series 2,2022,777000,100420,None
1,d31000cb-390d-4682-bee6-5b88a5704247,Mitsubishi Triton 2.4 GT Premium Plus 4WD Doub...,Mitsubishi,Triton,2018,499999,90000,None
2,ac7125e0-2044-4668-868b-e9672d6b7d40,W253 GLC250d AMG 2017 แท้ๆ ท็อปสุด option เต็ม...,Mercedes-Benz,GLC-Class,2017,898000,211737,None
3,84f4648f-1081-4d0c-aa27-1bd1a6195369,F30 320i Navi Lux แท้ ลงเล่ม 13 มือเดียว วิ่ง ...,BMW,Series 3,2013,479000,96600,None
4,c6e88266-0062-4346-bd10-c63c56b35f78,MG VS HEV 1.5 X Two tone ปี2025 สีขาว-ดำ ไมล์ ...,MG,VS HEV,2025,539999,11199,None


## 📦 Data Source 2: One2car (Multi-file Web Scraped Data Series)
- **ลักษณะข้อมูล**: ข้อมูลประกาศขายรถมือสองในไทยจากเว็บ One2car (สกัด 3 ช่วงเวลา)
- **รูปแบบไฟล์**: `one2car-11-2.csv`, `one2car-11-3.csv`, `one2car-11-4.csv` (CSV Multi-file Series)
- **ความซับซ้อนตามเกณฑ์อาจารย์**: `Text Data (Unstructured Regex)` + `ข้อมูลจากหลายไฟล์ (Multi-file Ingestion)`

In [7]:
# นำเข้า Data Source 2: One2car Multi-file
raw_one2car_files = sorted(glob.glob('../../01_Raw_Data/one2car/one2car-11-*.csv'))
print(f'พบไฟล์ One2car 3 ช่วงเวลา: {[os.path.basename(f) for f in raw_one2car_files]}')

def standardize_scraped_columns(df):
    rename_map = {
        'data': 'car_title',
        'data2': 'description',
        'data3': 'mileage',
        'data4': 'location',
        'data6': 'car_model',
        'data16': 'transmission'
    }
    return df.rename(columns=rename_map)

df_list = [standardize_scraped_columns(pd.read_csv(f)) for f in raw_one2car_files]
df_one2car = pd.concat(df_list, ignore_index=True)

print(f'✅ [Data Source 2: One2car Multi-file] โหลดและ Concat สำเร็จ: {len(df_one2car):,} แถว | {len(df_one2car.columns)} คอลัมน์')
df_one2car[['car_title', 'price', 'mileage', 'location', 'transmission']].head(5)

พบไฟล์ One2car 3 ช่วงเวลา: ['one2car-11-2.csv', 'one2car-11-3.csv', 'one2car-11-4.csv']
✅ [Data Source 2: One2car Multi-file] โหลดและ Concat สำเร็จ: 4,190 แถว | 15 คอลัมน์


,car_title,price,mileage,location,transmission
0,2015 Honda City 1.5 (ปี 14-18) SV+ Sedan - SV,"269,000 บาท",170 - 175K กม.,กรุงเทพมหานคร,เกียร์อัตโนมัติ
1,2023 Honda City 1.0 (ปี 19-26) SV Sedan,"359,000 บาท",30 - 35K กม.,สมุทรปราการ,เกียร์อัตโนมัติ
2,2025 BMW 220i 2.0 F44 (ปี 20-27) Gran M Sport ...,"1,250,000 บาท",30 - 35K กม.,สมุทรปราการ,เกียร์อัตโนมัติ
3,2022 Toyota HILUX REVO 2.4 Double Cab Z Editio...,"449,000 บาท",110 - 115K กม.,กรุงเทพมหานคร,เกียร์อัตโนมัติ
4,2015 Honda City 1.5 (ปี 14-18) SV Sedan,"259,000 บาท",20 - 25K กม.,กรุงเทพมหานคร,เกียร์อัตโนมัติ


## 📦 Data Source 3: US Used Car Sales (Historical Transaction Log Data Source)
- **ลักษณะข้อมูล**: สถิติธุรกรรมการขายจริง และระยะเวลาถือครองสต๊อก (`days_on_lot`) ย้อนหลัง
- **รูปแบบไฟล์**: `used_car_sales.csv` (CSV Transaction Log)
- **ความซับซ้อนตามเกณฑ์อาจารย์**: `Transaction Log Data Source`

In [8]:
# นำเข้า Data Source 3: US Used Car Sales
df_us_sales = pd.read_csv('../../01_Raw_Data/us-usecar/used_car_sales.csv')
print(f'✅ [Data Source 3: US Sales Log] โหลดสำเร็จ: {len(df_us_sales):,} แถว | {len(df_us_sales.columns)} คอลัมน์')
df_us_sales[['Make', 'Model', 'Year', 'pricesold', 'yearsold', 'Mileage']].head(5)

✅ [Data Source 3: US Sales Log] โหลดสำเร็จ: 122,144 แถว | 13 คอลัมน์


,Make,Model,Year,pricesold,yearsold,Mileage
0,Ford,Mustang,1988,7500,2020,84430
1,Replica/Kit Makes,Jaguar Beck Lister,1958,15000,2019,0
2,Jaguar,XJS,1995,8750,2020,55000
3,Ford,Mustang,1968,11600,2019,97200
4,Porsche,911,2002,44000,2019,40703
